<a href="https://colab.research.google.com/github/SyedaR06/CIS3120/blob/main/CIS3120_Module11_Pandas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CIS 3120 — Module 11: Pandas for Data Analysis

**CIS 3120: Programming for Analytics**  
Baruch College, Zicklin School of Business  
Professor Patrick Slattery

*This notebook is available at [bit.ly/cis3120_module11](https://bit.ly/cis3120_module11)*

---

This notebook spans **two class sessions**:

- **Class A** — Pandas Data Structures and Exploratory Analysis
- **Class B** — Merging DataFrames and Visualization

The dataset used throughout is the **NYC Department of Health Restaurant Inspection** data,
available at no cost from NYC Open Data.

**Before Class A:** Download the restaurant inspection CSV from the NYC Open Data site
and have it ready to upload to your Colab session.
NYC Open Data — DOHMH Restaurant Inspections:
https://data.cityofnewyork.us/Health/DOHMH-New-York-City-Restaurant-Inspection-Results/43nn-pn8j
  

**How to use this notebook:** Work through each section in order. When you reach
an **✏️ Exercise** block, write your answer in the cell immediately below the task
description. The cell begins with a `# ── TODO ──` comment and a few blank lines
for your code.

---

# Class A — Pandas Data Structures and Exploratory Analysis

## Learning Objectives

By the end of this session you will be able to:

1. Explain what a Pandas Series and a DataFrame are and how they relate to each other
2. Load a CSV file into a DataFrame and inspect its structure
3. Convert columns to appropriate data types (numeric, datetime, categorical)
4. Select, filter, rename, sort, and transform DataFrame rows and columns
5. Aggregate data using `groupby()`, `agg()`, and `pivot_table()`


## A.1 Setup and Imports

Run this cell first. It imports every library used in Class A.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

print("pandas version:", pd.__version__)
print("numpy version:", np.__version__)


## A.2 Pandas Data Structures

### A.2.1 The Series

A **Series** is a one-dimensional labeled array. Every element has a position
(the default integer index) and can also have a named label.

A Series can be created from a Python list, a dictionary, a NumPy array, or a
scalar value.


In [ ]:
# Series from a list — index defaults to 0, 1, 2, ...
inspection_scores = pd.Series([9, 13, 27, 4, 0])
print(inspection_scores)


In [ ]:
# Series from a dictionary — keys become the index labels
grade_counts = pd.Series({'A': 1842, 'B': 304, 'C': 97, 'Z': 51})
print(grade_counts)


In [ ]:
# Access a single element by label
print("Grade A count:", grade_counts['A'])

# Arithmetic works element-wise — equivalent to a NumPy array
print()
print("Proportions:")
print(grade_counts / grade_counts.sum())


### A.2.2 The DataFrame

A **DataFrame** is a two-dimensional labeled table — rows and columns.
Each column is a Series that shares the same row index.

You can think of a DataFrame as the Python equivalent of a spreadsheet or a
SQL table.


In [ ]:
# DataFrame from a dictionary of lists
# Each key becomes a column name; each list becomes the column values
sample_data = {
    'restaurant': ['Peking House', 'La Palapa', 'Corner Bagel', 'Spice Route'],
    'borough':    ['MANHATTAN',   'BROOKLYN',  'QUEENS',       'BRONX'],
    'score':      [9,             13,          27,             4],
    'grade':      ['A',           'A',         'B',            'A']
}

sample_df = pd.DataFrame(sample_data)
print(sample_df)


In [ ]:
# Each column is a Series
print(type(sample_df['score']))
print()
print(sample_df['score'])


---

### ✏️ Exercise A.2 — Build a DataFrame

**Task:** Create a DataFrame called `inspectors_df` with the following data.
Use a dictionary of lists as the source.

| inspector_id | borough    | inspections |
|---|---|---|
| 1001         | MANHATTAN  | 214         |
| 1002         | BROOKLYN   | 198         |
| 1003         | QUEENS     | 231         |
| 1004         | BRONX      | 177         |
| 1005         | STATEN IS  | 89          |

Then print the DataFrame and confirm it has 5 rows and 3 columns using `.shape`.


In [ ]:
# ── TODO ──────────────────────────────────────────────────────────────────
# Create inspectors_df here








---



## A.3 Loading the NYC Restaurant Inspection Data

### Choose UPLOAD or API

Choose one of the following two sub-sections to access the NYC Board of Health restaurant inspection data from NYC Open Data:  
- A3.UPLOAD: downloads the file from NYC then uploads to Colab  
- A3.API: reads the file from an NYC API into a Colab session  



###A.3.UPLOAD Uploading the `CSV` to Colab

1. Download the CSV from:  
   https://data.cityofnewyork.us/Health/DOHMH-New-York-City-Restaurant-Inspection-Results/43nn-pn8j  
   Click **Export → CSV** to download.

2. In Colab, click the **folder icon** in the left sidebar to open the Files panel.

3. Click the **upload icon** (page with an upward arrow) and select your downloaded CSV.

4. Note the filename shown in the Files panel — update the variable below if it differs.

In [ ]:
# Update this filename if your downloaded file has a different name
CSV_FILENAME = 'DOHMH_New_York_City_Restaurant_Inspection_Results.csv'


###A.3.API Pulling the `CSV` from an API

This notebook uses a dataset with [restaurant inspection results in NYC](https://data.cityofnewyork.us/Health/DOHMH-New-York-City-Restaurant-Inspection-Results/43nn-pn8j) which is available online from the City of New York.

Fetching a file to download in a Google Colab notebook results in the downloaded file residing in the '**sample_data/**' folder in the active Colab session.  That is a volatile copy which will not persist after the Colab session is closed.

This notebook fetchs the data using the [Linux curl command](https://www.geeksforgeeks.org/curl-command-in-linux-with-examples/) by executing the following command with the parameters **"-o"** and **"restaurant.csv"** to specify * file name to save the result:

In [ ]:
# Fetches the most recent dataset
!curl 'https://data.cityofnewyork.us/api/views/43nn-pn8j/rows.csv?accessType=DOWNLOAD' -o restaurant.csv
CSV_FILENAME = 'restaurant.csv'

###A.3.2 Reading the `CSV` Into a DataFrame

To be able to read and process this file within Python, the pandas library has a very convenient method `read_csv` which reads the file, and returns back a variable that contains its contents.  The following code creates a dataframe **restaurants** with the results of the Pandas **'.read_csv()'** function.

Using **'.read_csv()'** to read a CSV file (or TSV file), results in an object called a DataFrame, which is made up of rows and columns. DataFrame columns are accessed the same way as elements from a dictionary. Using the **restaurant** DataFrame object, the first five rows and the columns of data can be displayed with the **'.head(5)'** method:

In [ ]:
CSV_FILENAME = '/content/restaurant.csv'

In [ ]:
df = pd.read_csv(CSV_FILENAME)
print("Loaded", df.shape[0], "rows and", df.shape[1], "columns.")
df.head()


The **'.read_csv()'** method has many options.  See the [online documentation](http://pandas.pydata.org/pandas-docs/stable/generated/pandas.io.parsers.read_csv.html) for details.



---



## A.4 Inspecting the DataFrame

Before working with a dataset, always assess its structure: how many rows and
columns, what the column names are, what data types are present, and how many
values are missing.


In [ ]:
# Shape: (rows, columns)
print("Shape:", df.shape)


In [ ]:
# Column names and data types
print(df.dtypes)


In [ ]:
# Detailed summary: non-null counts, dtypes, memory usage
df.info()


In [ ]:
# Summary statistics for numeric columns
df.describe()


In [ ]:
# Transposing describe() makes it easier to read when there are many columns
df.describe().T


---

### ✏️ Exercise A.4 — Assess the Data

Answer the following questions using the methods above. Write a print statement
for each answer.

1. How many rows and columns does the dataset have?
2. What data type does pandas assign to the `SCORE` column?
3. What data type does pandas assign to the `INSPDATE` column?
4. How many non-null values are in the `GRADE` column?


In [ ]:
# ── TODO ──────────────────────────────────────────────────────────────────
# Answer each question with a print statement






## A.5 Data Type Conversion

Pandas often reads columns as `object` (string) even when the values are
numbers or dates. Converting to the correct type unlocks numeric operations,
date arithmetic, and efficient categorical storage.


In [ ]:
# Convert SCORE to numeric
# errors='coerce' turns any non-numeric value into NaN rather than raising an error
df['SCORE'] = pd.to_numeric(df['SCORE'], errors='coerce')
print("SCORE dtype after conversion:", df['SCORE'].dtype)
print("Sample values:", df['SCORE'].head().tolist())


In [ ]:
# Convert INSPDATE to datetime
# Pandas will infer the format; pass format='...' explicitly if it cannot
df['INSPDATE'] = pd.to_datetime(df['INSPECTION DATE'], errors='coerce')
print("INSPDATE dtype after conversion:", df['INSPDATE'].dtype)
print("Sample values:")
print(df['INSPDATE'].head())


In [ ]:
# Convert GRADE to categorical
# Categorical is useful for columns with a small, fixed set of values
# It also allows you to define an implicit order (e.g., A > B > C)
df['GRADE'] = df['GRADE'].astype('category')
print("GRADE dtype after conversion:", df['GRADE'].dtype)
print("Categories:", df['GRADE'].cat.categories.tolist())
print("Value counts:")
print(df['GRADE'].value_counts())


---

### ✏️ Exercise A.5 — Type Conversion

The `ZIPCODE` column was likely loaded as a float or object. Convert it to a
string type so it can be used as a label rather than a number.

*Hint:* `df['col'].astype(str)` converts a column to string type.  
After conversion, confirm the dtype and print the first five values.


In [ ]:
# ── TODO ──────────────────────────────────────────────────────────────────






## A.6 Selecting, Filtering, and Sorting

### A.6.1 Selecting Columns — `filter()`


In [ ]:
# Keep only the columns we care about for analysis
cols = ['CAMIS', 'DBA', 'BORO', 'ZIPCODE', 'CUISINE DESCRIPTION',
        'INSPDATE', 'SCORE', 'GRADE', 'VIOLATION CODE', 'CRITICAL FLAG']

df = df.filter(items=cols)
df.head(3)


### A.6.2 Renaming Columns — `rename()`

Long or all-caps column names are harder to type. Rename key columns to
lowercase, underscore-separated names.


In [ ]:
df = df.rename(columns={
    'CAMIS':               'camis',
    'DBA':                 'name',
    'BORO':                'borough',
    'ZIPCODE':             'zipcode',
    'CUISINE DESCRIPTION': 'cuisine',
    'INSPDATE':            'insp_date',
    'SCORE':               'score',
    'GRADE':               'grade',
    'VIOLATION CODE':      'violation_code',
    'CRITICAL FLAG':       'critical_flag'
})
df.head(3)


### A.6.3 Filtering Rows — `query()`

`query()` accepts a string expression similar to a SQL WHERE clause.


In [ ]:
# All inspections in Manhattan with a score above 20
manhattan_high = df.query("borough == 'Manhattan' and score > 20")
print("Rows matching filter:", len(manhattan_high))
manhattan_high.head()


In [ ]:
# Restaurants that received a grade of B or C
b_or_c = df.query("grade in ['B', 'C']")
print("B or C grade rows:", len(b_or_c))
b_or_c[['name', 'borough', 'score', 'grade']].head()


### A.6.4 Removing Duplicates — `drop_duplicates()`


In [ ]:
# Each restaurant (camis) may appear many times — once per inspection.
# To get one row per restaurant, keep only the first occurrence of each camis.
restaurants = df.drop_duplicates(subset=['camis'])
print("Unique restaurants:", len(restaurants))


### A.6.5 Sorting — `sort_values()`


In [ ]:
# Worst scores (highest number = more violations) at the top
df.sort_values(by='score', ascending=False)[['name', 'borough', 'score', 'grade']].head(10)


### A.6.6 Defining New Columns — `assign()` and `apply()`

Use `assign()` to add a new column derived from existing ones.  
Use `apply()` when the derivation requires row-level logic.


In [ ]:
# Classify each inspection as 'Pass' or 'Review' based on score threshold
# assign() returns a new DataFrame — reassign to df to keep the change
df = df.assign(
    result=df['score'].apply(lambda s: 'Pass' if pd.notna(s) and s < 14 else 'Review')
)
print(df['result'].value_counts())


---

### ✏️ Exercise A.6 — Filter, Sort, and Transform

Complete each task in its own cell.

**Task 1.** Filter `df` to return only rows where `cuisine` is `'Chinese'` and
`grade` is `'A'`. Store the result as `chinese_a`. Print the number of rows.

**Task 2.** Sort `chinese_a` by `score` ascending and display the top 5 rows,
showing only `name`, `borough`, `score`, and `grade`.

**Task 3.** Add a column called `score_band` to `df` that contains:
- `'Low'` when score is less than 14
- `'Medium'` when score is 14–27
- `'High'` when score is 28 or above
- `'Unknown'` when score is NaN

*Hint:* Write a helper function and pass it to `apply()`.


In [ ]:
# ── TODO Task 1 ───────────────────────────────────────────────────────────






In [ ]:
# ── TODO Task 2 ───────────────────────────────────────────────────────────






In [ ]:
# ── TODO Task 3 ───────────────────────────────────────────────────────────






## A.7 Aggregation

### A.7.1 `agg()` — Apply Functions to a Column


In [ ]:
# Summary statistics for the score column
df['score'].agg(['count', 'mean', 'median', 'min', 'max'])


### A.7.2 `groupby()` — Split, Apply, Combine

`groupby()` splits the DataFrame into groups, applies a function to each group,
and combines the results. This is the Pandas implementation of the
split-apply-combine pattern.


In [ ]:
# Mean score by borough
df.groupby('borough')['score'].mean().round(2)


In [ ]:
# Multiple aggregations at once using agg()
# For each borough: mean score, max score, and number of inspections
df.groupby('borough').agg(
    mean_score  = ('score', 'mean'),
    max_score   = ('score', 'max'),
    inspections = ('score', 'count')
).round(2)


In [ ]:
# Lambda inside agg() — count distinct values
df.groupby('borough').agg(
    unique_restaurants=('camis', lambda x: x.nunique())
)


### A.7.3 `pivot_table()` — Cross-Tabulation

A pivot table summarizes data across two categorical dimensions at once.


In [ ]:
# Number of inspections per grade per borough
pivot = pd.pivot_table(
    df,
    values='camis',
    index='borough',
    columns='grade',
    aggfunc='count',
    fill_value=0
)
pivot


In [ ]:
# Drop columns that represent missing or irrelevant grades
# (adjust the list based on what appears in your data)
grades_to_keep = ['A', 'B', 'C']
pivot_clean = pivot[[g for g in grades_to_keep if g in pivot.columns]]
pivot_clean


---

### ✏️ Exercise A.7 — Aggregation

**Task 1.** Using `groupby()`, compute the **median** score for each `cuisine`
type. Sort the result from highest median score to lowest and display the top 10.

**Task 2.** Build a pivot table that shows the **mean score** for each
combination of `borough` (rows) and `score_band` (columns).  
Use `fill_value=0`.


In [ ]:
# ── TODO Task 1 ───────────────────────────────────────────────────────────






In [ ]:
# ── TODO Task 2 ───────────────────────────────────────────────────────────






---

# Class B — Merging DataFrames and Visualization

## Learning Objectives

By the end of this session you will be able to:

1. Combine DataFrames using `pd.merge()`, `pd.concat()`, and `DataFrame.join()`
2. Distinguish inner, outer, left, and right joins and choose the appropriate type
3. Produce bar charts, line plots, and subplots using Matplotlib
4. Connect `groupby()` output directly to a `.plot()` call
5. Produce a statistical summary plot using Seaborn


## B.1 Setup

If you are starting a fresh Colab session for Class B, run the setup cell from
**A.1** and re-run the data loading and preparation cells from Class A
(sections A.3 through A.6) before continuing here.

The cell below imports the additional library needed for Class B.


In [ ]:
import seaborn as sns
print("seaborn version:", sns.__version__)


## B.2 Merging and Combining DataFrames

### B.2.1 Why Merging Matters

Real-world analytics data rarely arrives in a single table. Borough boundaries,
inspector assignments, business license records, and inspection results may all
live in separate files. Merging lets you assemble a unified view.

The mental model is identical to a SQL JOIN: two tables share a common key
column, and merge() uses that key to align their rows.

### B.2.2 A Second Dataset — Borough Reference Table

To demonstrate merging, we create a small reference DataFrame that maps each
borough name (as it appears in the inspection data) to additional attributes:
its official borough code, the county name, and its approximate population.


In [ ]:
# Borough reference table — created inline (no second upload needed)
borough_ref = pd.DataFrame({
    'borough':    ['Manhattan', 'Brooklyn', 'QueensS', 'Bronx', 'Staten Island'],
    'boro_code':  [1,           3,          4,        2,       5],
    'county':     ['New York',  'Kings',    'Queens', 'Bronx', 'Richmond'],
    'population': [1_629_000,   2_561_000,  2_253_000, 1_379_000, 476_000]
})
print(borough_ref)


Notice that the inspection data uses `'STATEN IS'` as a short form while the
reference table uses `'STATEN ISLAND'`. This deliberate mismatch lets us
observe how different join types handle unmatched keys.


### B.2.3 `pd.merge()` — Inner Join

An **inner join** returns only rows where the key value exists in **both**
DataFrames. Rows with no match are dropped.


In [ ]:
# Get one row per borough from the inspection data (borough name + mean score)
borough_scores = (df.groupby('borough', as_index=False)
                    .agg(mean_score=('score', 'mean'),
                         inspections=('camis', 'count'))
                    .round(2))
print("Borough scores table:")
print(borough_scores)


In [ ]:
# Inner join — only boroughs present in BOTH tables appear in the result
inner_merged = pd.merge(borough_scores, borough_ref, on='borough', how='inner')
print("Inner join — rows:", len(inner_merged))
inner_merged


### B.2.4 `pd.merge()` — Outer Join

A **full outer join** keeps every row from both DataFrames. Where a key exists
in only one table, the columns from the other table are filled with NaN.


In [ ]:
outer_merged = pd.merge(borough_scores, borough_ref, on='borough', how='outer')
print("Outer join — rows:", len(outer_merged))
outer_merged


In [ ]:
# The '0' / '' mismatch causes two rows in the outer join:
# one with inspection data but no reference data, and one with reference data but
# no inspection data. NaN marks whichever side had no match.


### B.2.5 Left and Right Joins

A **left join** keeps every row from the left DataFrame and matches what it can
from the right. A **right join** is the mirror image.


In [ ]:
# Left join — all boroughs from borough_scores are preserved
left_merged = pd.merge(borough_scores, borough_ref, on='borough', how='left')
print("Left join — rows:", len(left_merged))
left_merged


### B.2.6 Merging on Different Column Names

When the key column has a different name in each DataFrame, use `left_on` and
`right_on` instead of `on`.


In [ ]:
# Rename the borough column in the reference table to simulate a naming mismatch
borough_ref_renamed = borough_ref.rename(columns={'borough': 'boro_name'})

merged_diff_keys = pd.merge(
    borough_scores,
    borough_ref_renamed,
    left_on='borough',
    right_on='boro_name',
    how='inner'
)
merged_diff_keys


### B.2.7 `pd.concat()` — Stacking DataFrames

`concat()` stacks DataFrames along an axis — rows (axis=0, the default) or
columns (axis=1). Use it when the DataFrames share the same schema and you
want to combine them into one.


In [ ]:
# Split the inspection data into two halves and then reassemble
half = len(df) // 2
first_half  = df.iloc[:half]
second_half = df.iloc[half:]

reassembled = pd.concat([first_half, second_half], axis=0, ignore_index=True)
print("Original rows:", len(df))
print("Reassembled rows:", len(reassembled))


In [ ]:
# concat() along axis=1 — append columns side by side
# Here we attach the numeric summary columns to the borough reference table
borough_ref_extended = pd.concat([borough_ref, borough_scores.set_index('borough')], axis=1)
borough_ref_extended


### B.2.8 `combine_first()` — Fill Gaps from a Second Source

`combine_first()` uses values from a second DataFrame to fill NaN positions
in the first. It is useful when two datasets cover the same entities but
with partial overlap.


In [ ]:
# Simulate two partial DataFrames covering the same restaurants
# (in practice these might come from two different exports of the same system)
partial_a = df[['camis', 'name', 'score']].iloc[:500].set_index('camis')
partial_b = df[['camis', 'name', 'score']].iloc[400:].set_index('camis')

# Introduce some NaN into partial_a to make the fill visible
partial_a.loc[partial_a.index[:50], 'score'] = np.nan

combined = partial_a.combine_first(partial_b)
print("NaN scores in partial_a:", partial_a['score'].isna().sum())
print("NaN scores after combine_first:", combined['score'].isna().sum())


---

### ✏️ Exercise B.2 — Merge and Combine

The cell below creates a second small reference table: a cuisine category
lookup that maps specific cuisine descriptions to a broader category label.

**Task 1.** Perform an **inner join** between `df` and `cuisine_categories`
on the `cuisine` column. Store the result as `df_with_category`.  
Print the number of rows in `df_with_category` and compare it to `len(df)`.
Explain in a comment why the row counts differ.

**Task 2.** Perform an **outer join** between `df` and `cuisine_categories`.  
How many rows have a NaN value in the `category` column? Print that count.

**Task 3.** Using `pd.concat()`, stack the `inspectors_df` you created in
Exercise A.2 with the following new rows. Reset the index afterward.

| inspector_id | borough   | inspections |
|---|---|---|
| 1006         | Manhattan | 203         |
| 1007         | Brooklyn  | 187         |


In [ ]:
# Reference table provided for this exercise
cuisine_categories = pd.DataFrame({
    'cuisine':  ['Chinese', 'Italian', 'Mexican', 'Japanese', 'American',
                 'Pizza', 'Caribbean', 'French', 'Indian', 'Thai'],
    'category': ['Asian',   'European', 'Latin',  'Asian',   'American',
                 'American','Latin',    'European','Asian',  'Asian']
})
print(cuisine_categories)


In [ ]:
# ── TODO Task 1 ───────────────────────────────────────────────────────────






In [ ]:
# ── TODO Task 2 ───────────────────────────────────────────────────────────






In [ ]:
# ── TODO Task 3 ───────────────────────────────────────────────────────────






## B.3 Visualization with Matplotlib

### B.3.1 The Matplotlib Model

Matplotlib organizes plots into three layers:

- **Figure** — the entire canvas (the window or saved image)
- **Axes** — a single plot area inside the figure (what you draw on)
- **Artists** — the actual visual elements: lines, bars, tick labels, etc.

For most purposes you will use the `plt.subplots()` function, which creates a
Figure and one or more Axes objects in one step.


In [ ]:
# The simplest possible plot — Pandas calls Matplotlib behind the scenes
df.groupby('borough')['score'].mean().plot(kind='bar')
plt.title('Mean Inspection Score by Borough')
plt.ylabel('Mean Score')
plt.xlabel('Borough')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()


### B.3.2 Controlling the Figure with `plt.subplots()`

Using `plt.subplots()` gives you explicit control over figure size and the
arrangement of multiple plots on one canvas.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))

borough_means = df.groupby('borough')['score'].mean().sort_values(ascending=False)
ax.bar(borough_means.index, borough_means.values, color='steelblue', edgecolor='white')

ax.set_title('Mean Inspection Score by Borough', fontsize=14, fontweight='bold')
ax.set_xlabel('Borough', fontsize=11)
ax.set_ylabel('Mean Score (lower is better)', fontsize=11)
ax.tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()


### B.3.3 Subplots — Multiple Plots on One Canvas

Use `plt.subplots(nrows, ncols)` to create a grid of Axes objects.


In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(13, 4))

# Left plot — inspection count by borough
inspection_counts = df.groupby('borough')['camis'].count().sort_values(ascending=False)
axes[0].bar(inspection_counts.index, inspection_counts.values, color='teal', edgecolor='white')
axes[0].set_title('Inspections by Borough')
axes[0].set_xlabel('Borough')
axes[0].set_ylabel('Number of Inspections')
axes[0].tick_params(axis='x', rotation=20)

# Right plot — mean score by borough
borough_means = df.groupby('borough')['score'].mean().sort_values(ascending=False)
axes[1].bar(borough_means.index, borough_means.values, color='steelblue', edgecolor='white')
axes[1].set_title('Mean Score by Borough')
axes[1].set_xlabel('Borough')
axes[1].set_ylabel('Mean Score')
axes[1].tick_params(axis='x', rotation=20)

plt.suptitle('NYC Restaurant Inspections — Borough Summary', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


### B.3.4 Line Plot — Inspections Over Time

A line plot is appropriate when the x-axis represents a time dimension and you
want to see trends.


In [ ]:
# Count inspections per month
monthly = (df.set_index('insp_date')
             .resample('ME')['camis']
             .count()
             .reset_index())
monthly.columns = ['month', 'inspections']

# Drop any implausible dates (e.g., year 1900 noise rows)
monthly = monthly[monthly['month'].dt.year > 2010]

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(monthly['month'], monthly['inspections'], color='steelblue', linewidth=1.5)
ax.set_title('Monthly Inspection Volume', fontsize=13)
ax.set_xlabel('Month')
ax.set_ylabel('Inspections')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()


### B.3.5 Grouped Bar Chart — Grade Distribution by Borough


In [ ]:
# Pivot: rows = borough, columns = grade, values = count
grade_pivot = pd.pivot_table(
    df.query("grade in ['A', 'B', 'C']"),
    values='camis',
    index='borough',
    columns='grade',
    aggfunc='count',
    fill_value=0
)

grade_pivot.plot(kind='bar', figsize=(10, 5), edgecolor='white')
plt.title('Grade Distribution by Borough')
plt.xlabel('Borough')
plt.ylabel('Number of Inspections')
plt.xticks(rotation=20, ha='right')
plt.legend(title='Grade')
plt.tight_layout()
plt.show()


## B.4 Statistical Visualization with Seaborn

Seaborn is a higher-level visualization library built on top of Matplotlib.
It is designed for statistical graphics and produces polished results with
less code than raw Matplotlib.

The key difference: Seaborn works directly with a tidy DataFrame and a grammar
of aesthetics (`x=`, `y=`, `hue=`), while Matplotlib works with raw arrays.


In [ ]:
# Box plot — score distribution by borough, one box per borough
# A box plot shows median, quartiles, and outliers simultaneously
fig, ax = plt.subplots(figsize=(10, 5))

# Filter out NaN scores and limit to the five main boroughs
plot_df = df.dropna(subset=['score']).query(
    "borough in ['Manhattan', 'Brooklyn', 'Queens', 'Bronx', 'Staten Island']"
)

sns.boxplot(
    data=plot_df,
    x='borough',
    y='score',
    palette='Blues',
    ax=ax
)

ax.set_title('Inspection Score Distribution by Borough', fontsize=13)
ax.set_xlabel('Borough')
ax.set_ylabel('Score')
plt.tight_layout()
plt.show()


In [ ]:
# Bar plot with error bars — Seaborn computes the mean and 95% CI automatically
fig, ax = plt.subplots(figsize=(10, 5))

sns.barplot(
    data=plot_df,
    x='borough',
    y='score',
    palette='muted',
    ax=ax
)

ax.set_title('Mean Inspection Score by Borough (with 95% CI)', fontsize=13)
ax.set_xlabel('Borough')
ax.set_ylabel('Mean Score')
plt.tight_layout()
plt.show()


---

### ✏️ Exercise B.3 / B.4 — Visualization

**Task 1 (Matplotlib).** Create a horizontal bar chart showing the **top 10
cuisine types by number of inspections**. Use `kind='barh'` on a `groupby`
result. Add a title and axis labels.

**Task 2 (Matplotlib — subplots).** Create a figure with two subplots
side by side:
- Left: mean score by `grade` (A, B, C only), as a bar chart
- Right: number of inspections by `grade` (A, B, C only), as a bar chart

**Task 3 (Seaborn).** Produce a Seaborn box plot showing the distribution of
`score` for each `grade` (A, B, C only). Color the boxes using the `palette`
of your choice. Add an appropriate title.


In [ ]:
# ── TODO Task 1 ───────────────────────────────────────────────────────────






In [ ]:
# ── TODO Task 2 ───────────────────────────────────────────────────────────






In [ ]:
# ── TODO Task 3 ───────────────────────────────────────────────────────────






## B.5 Mapping Geographic Data with Folium

### B.5.1 Why Folium

The visualizations produced by Matplotlib and Seaborn are static images. For
geographic data, an interactive map is usually a better choice: the reader can
pan, zoom, and click individual points to see details.

**Folium** is a Python library that wraps the Leaflet JavaScript mapping
library. It produces interactive HTML maps directly from Python code. Folium
works naturally with Pandas DataFrames: each row with a latitude and longitude
becomes a point on the map, and any other column in the row can be displayed
in a popup.

The NYC restaurant inspection dataset includes `Latitude` and `Longitude`
columns for every inspected restaurant. That makes it a natural fit for
folium.

### B.5.2 Installing and Importing Folium

Google Colab includes folium in its default environment. The install command
below is a safeguard in case the environment has changed; it completes
quickly if folium is already present.

In [ ]:
!pip install folium --quiet

import folium
from folium.plugins import MarkerCluster

print("folium version:", folium.__version__)

### B.5.3 The Simplest Possible Map

A folium map is created with `folium.Map(location=[lat, lon], zoom_start=n)`.
Displaying the map object in a notebook cell renders it inline. The map below
is centered on Times Square in Manhattan.

In [ ]:
# Center on Times Square; zoom level 11 shows most of New York City
m = folium.Map(location=[40.7580, -73.9855], zoom_start=11, tiles='CartoDB positron', min_zoom=2)
m

### B.5.4 Preparing Geographic Data from the CSV

The `df` used earlier in this notebook was reduced to ten columns in section
**A.6.1**. Neither `Latitude` nor `Longitude` survived that filter. Rather
than revise the upstream pipeline, load the raw CSV a second time into a
separate DataFrame dedicated to mapping. This pattern is common in practice:
different analyses often require different projections of the same source
data.

Three cleaning steps are required before plotting:

1. **Drop rows with missing coordinates.** A restaurant with no recorded
   latitude or longitude cannot be placed on a map.
2. **Drop rows where both coordinates are zero.** The value `(0, 0)` is a
   sentinel used by the data provider when the true location is unknown.
   Plotted literally, these points land in the Atlantic Ocean off the coast
   of Africa and distort the map's auto-zoom.
3. **Deduplicate by `CAMIS`.** Each restaurant may appear in many inspection
   rows. For mapping, one row per restaurant is sufficient.

In [ ]:
# Reload the raw CSV so latitude and longitude are available
map_df = pd.read_csv(CSV_FILENAME)

# Standardize the column names we need for mapping
map_df = map_df.rename(columns={
    'CAMIS':               'camis',
    'DBA':                 'name',
    'BORO':                'borough',
    'CUISINE DESCRIPTION': 'cuisine',
    'SCORE':               'score',
    'GRADE':               'grade',
    'Latitude':            'latitude',
    'Longitude':           'longitude',
})

# Keep only the columns needed for mapping
map_df = map_df.filter(items=['camis', 'name', 'borough', 'cuisine',
                              'score', 'grade', 'latitude', 'longitude'], tiles='CartoDB positron', min_zoom=2)

print("Rows before cleaning:", len(map_df))

# Step 1: drop rows with missing coordinates
map_df = map_df.dropna(subset=['latitude', 'longitude'])

# Step 2: drop rows with (0, 0) sentinel values
map_df = map_df.query('latitude != 0 and longitude != 0')

# Step 3: keep one row per restaurant
map_df = map_df.drop_duplicates(subset='camis').reset_index(drop=True)

print("Rows after cleaning:", len(map_df))
map_df.head()

### B.5.5 Adding Markers from a DataFrame

Each marker is placed individually with `folium.Marker(...).add_to(m)`.
Iterate over the DataFrame with `iterrows()` to produce one marker per row.

Rendering thousands of markers at once is slow in the browser. The example
below samples 200 restaurants. Later subsections show how to handle larger
datasets with `CircleMarker` and `MarkerCluster`.

In [ ]:
# A random sample keeps the map responsive
sample_df = map_df.sample(n=200, random_state=42)

m = folium.Map(location=[40.7580, -73.9855], zoom_start=11, tiles='CartoDB positron', min_zoom=2)

for _, row in sample_df.iterrows():
    folium.Marker(
        location=[row['latitude'], row['longitude']],
        popup=row['name']
    ).add_to(m)

m

### B.5.6 CircleMarker and Color Coding

`folium.CircleMarker` is a lighter-weight alternative to `folium.Marker`. It
renders a simple colored circle rather than a detailed pin icon, which makes
it appropriate for displaying hundreds of points at once.

Color can be used to encode an additional variable. The helper function
`grade_color()` below maps each inspection grade to a color:

| Grade | Color |
|---|---|
| A | green |
| B | orange |
| C | red |
| Other / missing | gray |

This turns the map into a two-dimensional visualization: location and grade
are both visible at a glance.

In [ ]:
def grade_color(grade):
    if grade == 'A':
        return 'green'
    if grade == 'B':
        return 'orange'
    if grade == 'C':
        return 'red'
    return 'gray'

# Manhattan typically has thousands of unique restaurants. Sample down
# so the map renders quickly in the browser.
manhattan_df = map_df.query("borough == 'Manhattan'")
n_sample = min(500, len(manhattan_df))
manhattan_df = manhattan_df.sample(n=n_sample, random_state=42)

m = folium.Map(location=[40.7831, -73.9712], zoom_start=12, tiles='CartoDB positron', min_zoom=2)

for _, row in manhattan_df.iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=4,
        color=grade_color(row['grade']),
        fill=True,
        fill_color=grade_color(row['grade']),
        fill_opacity=0.7,
        popup=row['name']
    ).add_to(m)

m

### B.5.7 Rich Popups with Restaurant Details

A popup argument can be any HTML string wrapped in a `folium.Popup` object.
This allows multiple fields from the DataFrame row to appear in a formatted
block when the marker is clicked.

The pattern below builds an HTML string for each row using an f-string and
then passes it to `folium.Popup(html, max_width=...)`.

In [ ]:
m = folium.Map(location=[40.7831, -73.9712], zoom_start=12, tiles='CartoDB positron', min_zoom=2)

for _, row in manhattan_df.iterrows():
    html = (
        f"<b>{row['name']}</b><br>"
        f"Cuisine: {row['cuisine']}<br>"
        f"Borough: {row['borough']}<br>"
        f"Score: {row['score']}<br>"
        f"Grade: {row['grade']}"
    )
    popup = folium.Popup(html, max_width=250)

    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=4,
        color=grade_color(row['grade']),
        fill=True,
        fill_color=grade_color(row['grade']),
        fill_opacity=0.7,
        popup=popup
    ).add_to(m)

m

### B.5.8 Saving the Map to an HTML File

The `save()` method writes the map to a standalone HTML file that can be
opened in any web browser, emailed to a colleague, or embedded in a website.

In Colab, the file appears in the file browser on the left side of the
screen. The second line below triggers a browser download of the file.

In [ ]:
m.save('manhattan_inspections.html')

# Optional: trigger a download in Colab
# from google.colab import files
# files.download('manhattan_inspections.html')

---

### ✏️ Exercise B.5 — Mapping

**Task 1.** Build a map of the 50 Manhattan restaurants with the highest
inspection scores. In this dataset, a higher score indicates more violations,
so these are the worst inspection results. Use `CircleMarker` in red, and
include each restaurant's name in the popup.

**Task 2.** Build a map of pizzerias in Brooklyn. Select rows where `borough`
equals `'BROOKLYN'` and `cuisine` equals `'Pizza'`. Color-code the markers by
grade using the `grade_color()` function from section B.5.6. Include the
restaurant name and the score in the popup.

**Task 3.** Build a map of 1,000 randomly sampled restaurants from all five
boroughs using `folium.plugins.MarkerCluster`. MarkerCluster groups nearby
markers at low zoom levels and expands them as the user zooms in. This is the
standard approach for mapping thousands of points without slowing the browser.
Use `MarkerCluster().add_to(m)` to create the cluster, then add each
`folium.Marker` to the cluster instead of directly to the map.

In [ ]:
# ── TODO Task 1 ───────────────────────────────────────────────────────────






In [ ]:
# ── TODO Task 2 ───────────────────────────────────────────────────────────






In [ ]:
# ── TODO Task 3 ───────────────────────────────────────────────────────────






## B.6 Capstone Exercise — End-to-End Analysis

This exercise brings together every skill from both class sessions.

**Scenario:** A food-safety advocacy group wants a borough-level summary
comparing the volume, quality, and trend of restaurant inspections across
NYC. You will build that summary from scratch.

---

**Step 1.** Start from `df`. Using `groupby()` and `agg()`, create a summary
DataFrame called `borough_summary` with the following columns:

| Column | Description |
|---|---|
| `borough` | Borough name |
| `restaurants` | Number of unique restaurants (`camis`) |
| `inspections` | Total number of inspection records |
| `mean_score` | Mean inspection score (rounded to 2 decimal places) |
| `pct_grade_a` | Percentage of inspections that received grade A (rounded to 1 decimal place) |

**Step 2.** Merge `borough_summary` with `borough_ref` (the reference table
from section B.2.2) using an inner join on `borough`. Store the result as
`borough_full`.

**Step 3.** Add a column called `inspections_per_1k` that represents the number
of inspections per 1,000 residents (inspections ÷ population × 1000), rounded
to 1 decimal place.

**Step 4.** Produce a single figure with three subplots arranged in one row:
- Subplot 1: `restaurants` by borough (bar chart)
- Subplot 2: `mean_score` by borough (bar chart)
- Subplot 3: `pct_grade_a` by borough (bar chart)

Use consistent borough ordering (sort by `mean_score` ascending) across all
three subplots. Add a figure-level title.

**Step 5.** Print `borough_full` sorted by `mean_score` ascending.


In [ ]:
# ── TODO Step 1 ───────────────────────────────────────────────────────────






In [ ]:
# ── TODO Step 2 ───────────────────────────────────────────────────────────






In [ ]:
# ── TODO Step 3 ───────────────────────────────────────────────────────────






In [ ]:
# ── TODO Step 4 ───────────────────────────────────────────────────────────






In [ ]:
# ── TODO Step 5 ───────────────────────────────────────────────────────────






---

## End of Module 11

**Skills practiced in this notebook:**

| Skill | Sections |
|---|---|
| Create and manipulate Pandas DataFrames | A.2, A.3 |
| Filter and slice data | A.6 |
| Aggregate with groupby and pivot tables | A.7 |
| Merge and join datasets | B.2 |
| Visualize with Matplotlib and Seaborn | B.3, B.4 |
| Create interactive maps from a DataFrame with Folium | B.5 |

**References**
- Wes McKinney, *Python for Data Analysis*, 3rd ed. — https://wesmckinney.com/book/
- Pandas documentation — https://pandas.pydata.org/docs/
- Matplotlib documentation — https://matplotlib.org/stable/
- Seaborn documentation — https://seaborn.pydata.org/
- Folium documentation — https://python-visualization.github.io/folium/
- NYC Open Data — Restaurant Inspections — https://data.cityofnewyork.us/Health/DOHMH-New-York-City-Restaurant-Inspection-Results/43nn-pn8j
